# 05 k-means on real data

Same algorithm as week 0, different dataset. We run k-means on the covertype sample and then
ask the uncomfortable question: **do the clusters mean anything?**

We have the species labels, so we can check.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

CLASS_NAMES = {1: "Spruce/Fir", 2: "Lodgepole Pine", 3: "Ponderosa Pine",
               4: "Cottonwood/Willow", 5: "Aspen", 6: "Douglas-fir",
               7: "Krummholz"}

RANDOM_STATE = 0

d = np.load("data/covertype_sample.npz")
X, y = d["X"], d["y"]
FEATURES = list(d["feature_names"])

### Standardise first

In [ ]:
Xs = StandardScaler().fit_transform(X)
print("after scaling: mean ~0, std ~1 for every feature")
print("  std range:", Xs.std(axis=0).min().round(4), "-", Xs.std(axis=0).max().round(4))

### Run k-means with k = 7

Seven, because we happen to know there are seven species.

In [ ]:
K = 7

t0 = time.perf_counter()
km = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(Xs)
seconds = time.perf_counter() - t0

labels = km.labels_

print(f"fitted in {seconds:.2f} s")
print(f"inertia            : {km.inertia_:,.0f}")
print(f"silhouette         : {silhouette_score(Xs, labels, sample_size=5000, random_state=RANDOM_STATE):.3f}")
print(f"adjusted Rand index: {adjusted_rand_score(y, labels):.3f}")

### Clusters

In [ ]:
ct = pd.crosstab(pd.Series(labels, name="cluster"),
                 pd.Series([CLASS_NAMES[int(c)] for c in y], name="species"))
ct

In [ ]:
# how much of each cluster is its most common species?
share = (ct.max(axis=1) / ct.sum(axis=1)).round(3)
pd.DataFrame({"cluster size": ct.sum(axis=1),
              "most common species": ct.idxmax(axis=1),
              "share of cluster": share})